# Ejemplar: Construir un modelo de bosque aleatorio


## **Introducción**


Mientras aprendes, los bosques aleatorios son algoritmos estadísticos de aprendizaje populares. Algunas de sus principales ventajas incluyen reducir la varianza, el sesgo y la probabilidad de sobreajuste.

Esta actividad es una continuación del proyecto que comenzaste modelando con árboles de decisión para una aerolínea. Aquí, entrenarás, ajustarás y evaluarás un modelo de bosque aleatorio usando datos de una hoja de cálculo de respuestas de encuestas de 129,880 clientes. Incluye puntos de datos como clase, distancia del vuelo y entretenimiento a bordo. Tu modelo de bosque aleatorio será utilizado para predecir si un cliente estará satisfecho con su experiencia de vuelo.

**Nota:** Debido a que este laboratorio usa un conjunto de datos real, este cuaderno primero requiere análisis exploratorio de datos, limpieza de datos y otras manipulaciones para prepararlo para el modelado.


## **Paso 1: Importaciones**


Importar las bibliotecas y módulos relevantes de Python, incluyendo las bibliotecas `numpy` y `pandas` para el procesamiento de datos; el paquete `pickle` para guardar el modelo; y la biblioteca `sklearn`, que contiene:
- El módulo `ensemble`, que tiene la función `RandomForestClassifier`
- El módulo `model_selection`, que tiene las funciones `train_test_split`, `PredefinedSplit`, y `GridSearchCV`
- El módulo `metrics`, que tiene las funciones `f1_score`, `precision_score`, `recall_score`, y `accuracy_score`


In [1]:
# Import `numpy`, `pandas`, `pickle`, and `sklearn`.
# Import the relevant functions from `sklearn.ensemble`, `sklearn.model_selection`, and `sklearn.metrics`.

### YOUR CODE HERE ###
 
import numpy as np
import pandas as pd

import pickle as pkl
 
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, PredefinedSplit, GridSearchCV
from sklearn.metrics import f1_score, precision_score, recall_score, accuracy_score

Como se muestra en esta celda, el conjunto de datos ha sido cargado automáticamente para ti. No necesitas descargar el archivo .csv, ni proporcionar más código, para acceder al conjunto de datos y continuar con este laboratorio. Por favor, continúa con esta actividad completando las siguientes instrucciones.


In [2]:
# RUN THIS CELL TO IMPORT YOUR DATA. 

### YOUR CODE HERE ###

air_data = pd.read_csv("Invistico_Airline.csv")

<details>
  <summary><h4><strong>Pista 1</strong></h4></summary>

La función `read_csv()` de la biblioteca `pandas` puede ser útil aquí.
 
</details>


Ahora, estás listo para comenzar a limpiar tus datos.


## **Paso 2: Limpieza de datos**


Para tener una idea de los datos, mostrar las primeras 10 filas.


In [3]:
# Display first 10 rows.

### YOUR CODE HERE ###

air_data.head(10)

,satisfaction,Customer Type,Age,Type of Travel,Class,Flight Distance,Seat comfort,Departure/Arrival time convenient,Food and drink,Gate location,...,Online support,Ease of Online booking,On-board service,Leg room service,Baggage handling,Checkin service,Cleanliness,Online boarding,Departure Delay in Minutes,Arrival Delay in Minutes
0,satisfied,Loyal Customer,65,Personal Travel,Eco,265,0,0,0,2,...,2,3,3,0,3,5,3,2,0,0.0
1,satisfied,Loyal Customer,47,Personal Travel,Business,2464,0,0,0,3,...,2,3,4,4,4,2,3,2,310,305.0
2,satisfied,Loyal Customer,15,Personal Travel,Eco,2138,0,0,0,3,...,2,2,3,3,4,4,4,2,0,0.0
3,satisfied,Loyal Customer,60,Personal Travel,Eco,623,0,0,0,3,...,3,1,1,0,1,4,1,3,0,0.0
4,satisfied,Loyal Customer,70,Personal Travel,Eco,354,0,0,0,3,...,4,2,2,0,2,4,2,5,0,0.0
5,satisfied,Loyal Customer,30,Personal Travel,Eco,1894,0,0,0,3,...,2,2,5,4,5,5,4,2,0,0.0
6,satisfied,Loyal Customer,66,Personal Travel,Eco,227,0,0,0,3,...,5,5,5,0,5,5,5,3,17,15.0
7,satisfied,Loyal Customer,10,Personal Travel,Eco,1812,0,0,0,3,...,2,2,3,3,4,5,4,2,0,0.0
8,satisfied,Loyal Customer,56,Personal Travel,Business,73,0,0,0,3,...,5,4,4,0,1,5,4,4,0,0.0
9,satisfied,Loyal Customer,22,Personal Travel,Eco,1556,0,0,0,3,...,2,2,2,4,5,3,4,2,30,26.0


<details>
  <summary><h4><strong>Pista 1</strong></h4></summary>

La función `head()` de la biblioteca `pandas` puede ser útil aquí.
 
</details>


Ahora, muestra los nombres de las variables y sus tipos de datos.


In [4]:
# Display variable names and types.

### YOUR CODE HERE ###

air_data.dtypes

satisfaction                             str
Customer Type                            str
Age                                    int64
Type of Travel                           str
Class                                    str
Flight Distance                        int64
Seat comfort                           int64
Departure/Arrival time convenient      int64
Food and drink                         int64
Gate location                          int64
Inflight wifi service                  int64
Inflight entertainment                 int64
Online support                         int64
Ease of Online booking                 int64
On-board service                       int64
Leg room service                       int64
Baggage handling                       int64
Checkin service                        int64
Cleanliness                            int64
Online boarding                        int64
Departure Delay in Minutes             int64
Arrival Delay in Minutes             float64
dtype: obj

<details>
  <summary><h4><strong>Pista 1</strong></h4></summary>

DataFrames tienen un atributo que muestra los nombres de variables y los tipos de datos en un solo resultado.
 
</details>


**Pregunta:** ¿Qué observas acerca de las diferencias en los tipos de datos entre las variables incluidas en los datos?

Hay tres tipos de variables incluidas en los datos: int64, float64 y object. Las variables de objeto son satisfacción, tipo de cliente, tipo de viaje y clase.


A continuación, para entender el tamaño del conjunto de datos, identificar el número de filas y el número de columnas.


In [5]:
# Identify the number of rows and the number of columns.

### YOUR CODE HERE ###

air_data.shape

(129880, 22)

<details>
  <summary><h4><strong>Pista 1</strong></h4></summary>

Hay un método en la biblioteca `pandas` que devuelve el número de filas y el número de columnas en un solo resultado.

</details>


Ahora, verifica si hay valores faltantes en las filas de los datos. Comienza con .isna() para obtener booleanos que indican si cada valor en los datos falta. Luego, usa .any(axis=1) para obtener booleanos que indican si hay valores faltantes a lo largo de las columnas en cada fila. Finalmente, usa .sum() para obtener el número de filas que contienen valores faltantes.


In [6]:
# Get Booleans to find missing values in data.
# Get Booleans to find missing values along columns.
# Get the number of rows that contain missing values.

### YOUR CODE HERE ###

air_data.isna().any(axis=1).sum()

np.int64(393)

**Pregunta:** ¿Cuántas filas de datos tienen valores faltantes?**

Hay 393 filas con valores faltantes.


Drop the rows with missing values. This is an important step in data cleaning, as it makes the data more useful for analysis and regression. Then, save the resulting pandas DataFrame in a variable named `air_data_subset`.


In [7]:
# Drop missing values.
# Save the DataFrame in variable `air_data_subset`.

### YOUR CODE HERE ###

air_data_subset = air_data.dropna(axis=0)

<details>
<summary><h4><strong>Pista 1</strong></h4></summary>

La función `dropna()` es útil aquí.
</details>


<details>
<summary><h4><strong>Pista 2</strong></h4></summary>

El parámetro axis pasado a esta función debe establecerse en 0 (si deseas eliminar filas que contienen valores faltantes) o 1 (si deseas eliminar columnas que contienen valores faltantes).
</details>


A continuación, muestra las primeras 10 filas para examinar el subconjunto de datos.


In [8]:
# Display the first 10 rows.

### YOUR CODE HERE ###

air_data_subset.head(10)

,satisfaction,Customer Type,Age,Type of Travel,Class,Flight Distance,Seat comfort,Departure/Arrival time convenient,Food and drink,Gate location,...,Online support,Ease of Online booking,On-board service,Leg room service,Baggage handling,Checkin service,Cleanliness,Online boarding,Departure Delay in Minutes,Arrival Delay in Minutes
0,satisfied,Loyal Customer,65,Personal Travel,Eco,265,0,0,0,2,...,2,3,3,0,3,5,3,2,0,0.0
1,satisfied,Loyal Customer,47,Personal Travel,Business,2464,0,0,0,3,...,2,3,4,4,4,2,3,2,310,305.0
2,satisfied,Loyal Customer,15,Personal Travel,Eco,2138,0,0,0,3,...,2,2,3,3,4,4,4,2,0,0.0
3,satisfied,Loyal Customer,60,Personal Travel,Eco,623,0,0,0,3,...,3,1,1,0,1,4,1,3,0,0.0
4,satisfied,Loyal Customer,70,Personal Travel,Eco,354,0,0,0,3,...,4,2,2,0,2,4,2,5,0,0.0
5,satisfied,Loyal Customer,30,Personal Travel,Eco,1894,0,0,0,3,...,2,2,5,4,5,5,4,2,0,0.0
6,satisfied,Loyal Customer,66,Personal Travel,Eco,227,0,0,0,3,...,5,5,5,0,5,5,5,3,17,15.0
7,satisfied,Loyal Customer,10,Personal Travel,Eco,1812,0,0,0,3,...,2,2,3,3,4,5,4,2,0,0.0
8,satisfied,Loyal Customer,56,Personal Travel,Business,73,0,0,0,3,...,5,4,4,0,1,5,4,4,0,0.0
9,satisfied,Loyal Customer,22,Personal Travel,Eco,1556,0,0,0,3,...,2,2,2,4,5,3,4,2,30,26.0


Confirma que no contiene valores faltantes.


In [9]:
# Count of missing values.

### YOUR CODE HERE ###

air_data_subset.isna().sum()

satisfaction                         0
Customer Type                        0
Age                                  0
Type of Travel                       0
Class                                0
Flight Distance                      0
Seat comfort                         0
Departure/Arrival time convenient    0
Food and drink                       0
Gate location                        0
Inflight wifi service                0
Inflight entertainment               0
Online support                       0
Ease of Online booking               0
On-board service                     0
Leg room service                     0
Baggage handling                     0
Checkin service                      0
Cleanliness                          0
Online boarding                      0
Departure Delay in Minutes           0
Arrival Delay in Minutes             0
dtype: int64

<details>
<summary><h4><strong>Pista 1</strong></h4></summary>

Puedes usar `.isna().sum()` para obtener el número de valores faltantes para cada variable.

</details>


A continuación, convierte las características categóricas en características de indicador (codificación one-hot).

**Nota:** El argumento `drop_first` puede mantenerse como predeterminado (`False`) durante la codificación one-hot para modelos de bosque aleatorio, por lo que no es necesario especificarlo. Además, la variable objetivo, `satisfaction`, no necesita ser codificada y será extraída en un paso posterior.


In [10]:
# Convert categorical features to one-hot encoded features.

### YOUR CODE HERE ###

air_data_subset_dummies = pd.get_dummies(air_data_subset, 
                                         columns=['Customer Type','Type of Travel','Class'])

<details>
<summary><h4><strong>Pista 1</strong></h4></summary>

Puedes usar la función `pd.get_dummies()` para convertir variables categóricas en variables codificadas en una sola posición.
</details>


**Pregunta:** ¿Por qué es necesario convertir los datos categóricos en variables dummy?

Es necesario porque la implementación de sklearn de `RandomForestClassifier()` requiere que las características categóricas estén codificadas en numérico, lo cual se puede hacer usando variables dummy o codificación one-hot.


A continuación, muestre las primeras 10 filas para revisar el `air_data_subset_dummies`.


In [11]:
# Display the first 10 rows.

### YOUR CODE HERE ###

air_data_subset_dummies.head(10)

,satisfaction,Age,Flight Distance,Seat comfort,Departure/Arrival time convenient,Food and drink,Gate location,Inflight wifi service,Inflight entertainment,Online support,...,Online boarding,Departure Delay in Minutes,Arrival Delay in Minutes,Customer Type_Loyal Customer,Customer Type_disloyal Customer,Type of Travel_Business travel,Type of Travel_Personal Travel,Class_Business,Class_Eco,Class_Eco Plus
0,satisfied,65,265,0,0,0,2,2,4,2,...,2,0,0.0,True,False,False,True,False,True,False
1,satisfied,47,2464,0,0,0,3,0,2,2,...,2,310,305.0,True,False,False,True,True,False,False
2,satisfied,15,2138,0,0,0,3,2,0,2,...,2,0,0.0,True,False,False,True,False,True,False
3,satisfied,60,623,0,0,0,3,3,4,3,...,3,0,0.0,True,False,False,True,False,True,False
4,satisfied,70,354,0,0,0,3,4,3,4,...,5,0,0.0,True,False,False,True,False,True,False
5,satisfied,30,1894,0,0,0,3,2,0,2,...,2,0,0.0,True,False,False,True,False,True,False
6,satisfied,66,227,0,0,0,3,2,5,5,...,3,17,15.0,True,False,False,True,False,True,False
7,satisfied,10,1812,0,0,0,3,2,0,2,...,2,0,0.0,True,False,False,True,False,True,False
8,satisfied,56,73,0,0,0,3,5,3,5,...,4,0,0.0,True,False,False,True,True,False,False
9,satisfied,22,1556,0,0,0,3,2,0,2,...,2,30,26.0,True,False,False,True,False,True,False


Luego, verifica las variables de air_data_subset_dummies.


In [12]:
# Display variables.

### YOUR CODE HERE ###

air_data_subset_dummies.dtypes

satisfaction                             str
Age                                    int64
Flight Distance                        int64
Seat comfort                           int64
Departure/Arrival time convenient      int64
Food and drink                         int64
Gate location                          int64
Inflight wifi service                  int64
Inflight entertainment                 int64
Online support                         int64
Ease of Online booking                 int64
On-board service                       int64
Leg room service                       int64
Baggage handling                       int64
Checkin service                        int64
Cleanliness                            int64
Online boarding                        int64
Departure Delay in Minutes             int64
Arrival Delay in Minutes             float64
Customer Type_Loyal Customer            bool
Customer Type_disloyal Customer         bool
Type of Travel_Business travel          bool
Type of Tr

**Pregunta:** ¿Qué cambios observas después de convertir los datos de cadenas en variables dummy?

Todos los siguientes cambios podrían ser observados:

- Tipo de Cliente  -->  Tipo de Cliente_Cliente Leal y Tipo de Cliente_Cliente No Leal
- Tipo de Viaje -->  Tipo de Viaje_Viaje de Negocios y Tipo de Viaje_Viaje Personal
- Clase          --> Clase_Negocios, Clase_Eco, Clase_Eco Plus


## **Paso 3: Construcción del modelo**


El primer paso para construir tu modelo es separar las etiquetas (y) de las características (X).


In [13]:
# Separate the dataset into labels (y) and features (X).

### YOUR CODE HERE ###

y = air_data_subset_dummies["satisfaction"]
X = air_data_subset_dummies.drop("satisfaction", axis=1)

<details>
<summary><h4><strong>Pista 1</strong></h4></summary>

Guarda las etiquetas (los valores en la columna `satisfaction`) como `y`.

Guarda las características como `X`. 

</details>


<details>
<summary><h4><strong>Pista 2</strong></h4></summary>

Para obtener las características, elimina la columna `satisfaction` del DataFrame.

</details>


Una vez separado, divide los datos en conjuntos de entrenamiento, validación y prueba.


In [14]:
# Separate into train, validate, test sets.

### YOUR CODE HERE ###

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.25, random_state = 0)
X_tr, X_val, y_tr, y_val = train_test_split(X_train, y_train, test_size = 0.25, random_state = 0)


<details>
<summary><h4><strong>Pista 1</strong></h4></summary>

Utilice la función `train_test_split()` dos veces para crear conjuntos de entrenamiento/validación/prueba, pasando `random_state` para resultados reproducibles.

</details>


<details>
<summary><h4><strong>Pista 1</strong></h4></summary>

Divide `X`, `y` para obtener `X_train`, `X_test`, `y_train`, `y_test`. Establece el argumento `test_size` a la proporción de puntos de datos que deseas seleccionar para prueba. 

Divide `X_train`, `y_train` para obtener `X_tr`, `X_val`, `y_tr`, `y_val`. Establece el argumento `test_size` a la proporción de puntos de datos que deseas seleccionar para validación. 

</details>


### Ajustar el modelo

Ahora, ajuste y ajuste un modelo de bosque aleatorio con un conjunto de validación separado. Comience por determinar un conjunto de hiperparámetros para ajustar el modelo usando GridSearchCV.


In [15]:
# Determine set of hyperparameters.

### YOUR CODE HERE ###

cv_params = {'n_estimators' : [50,100], 
              'max_depth' : [10,50],        
              'min_samples_leaf' : [0.5,1], 
              'min_samples_split' : [0.001, 0.01],
              'max_features' : ["sqrt"], 
              'max_samples' : [.5,.9]}

<details>
<summary><h4><strong>Pista 1</strong></h4></summary>

Crea un diccionario `cv_params` que asigna a cada nombre de hiperparámetro una lista de valores. La búsqueda en cuadrícula que realices establecerá el hiperparámetro a cada valor posible, según lo especificado, y determinará qué valor es óptimo.

</details>


<details>
<summary><h4><strong>Pista 2</strong></h4></summary>

Los principales hiperparámetros aquí incluyen `'n_estimators', 'max_depth', 'min_samples_leaf', 'min_samples_split', 'max_features', y 'max_samples'`. Estos serán las claves en el diccionario `cv_params`.

</details>


A continuación, crea una lista de índices de división.


In [16]:
# Create list of split indices.

### YOUR CODE HERE ###

split_index = [0 if x in X_val.index else -1 for x in X_train.index]
custom_split = PredefinedSplit(split_index)

In [18]:
custom_split

PredefinedSplit(test_fold=array([-1, -1, ..., -1, -1], shape=(97115,)))

<details>
<summary><h4><strong>Pista 1</strong></h4></summary>

Utiliza comprensión de listas, iterando sobre los índices de `X_train`. La lista puede consistir en 0s para indicar puntos de datos que deben ser tratados como datos de validación y -1s para indicar puntos de datos que deben ser tratados como datos de entrenamiento.

</details>


<details>
<summary><h4><strong>Pista 2</strong></h4></summary>

Use `PredfinedSplit()`, passing in `split_index`, saving the output as `custom_split`. Esto servirá como una división personalizada que identificará qué puntos de datos del conjunto de entrenamiento deben tratarse como datos de validación durante GridSearch.

</details>


Ahora, instancia tu modelo.


In [19]:
# Instantiate model.

### YOUR CODE HERE ### 

rf = RandomForestClassifier(random_state=0)

<details>
<summary><h4><strong>Pista 1</strong></h4></summary>

Utiliza `RandomForestClassifier()`, especificando el argumento `random_state` para resultados reproducibles. Esto te ayudará a instanciar un modelo de bosque aleatorio, `rf`.

</details>


A continuación, use GridSearchCV para buscar entre los parámetros especificados.


In [21]:
# Search over specified parameters.

### YOUR CODE HERE ### 

rf_val = GridSearchCV(
    rf,              # Modelo a estudiar
    cv_params,       # Paramatros a estudiar
    cv=custom_split, # Estrategia de validación cruzada
    refit='f1',      # refit the best estimator with the best parameters # refinación de los mejores parámetros 
    n_jobs = -1,     # usar los núcleos disponibles
    verbose = 1)     # nivel de detalle de la salida de la consola

In [ ]:
# Fit model to training data.
rf_val.fit(X_train, y_train)
# funcion para obtener el mejor modelo
def get_best_model(rf_val):
    """
    Function to return the best model from a GridSearchCV object.
    
    Parameters:
    rf_val (GridSearchCV): The GridSearchCV object containing the fitted models.
    
    Returns:
    best_model: The best model found during the grid search.

    Funcion objetivo: Retorna el mejor modelo de un objeto GridSearchCV.

    Parámetros:
    rf_val (GridSearchCV): El objeto GridSearchCV que contiene los modelos ajustados
    Retorna:
    best_model: El mejor modelo encontrado durante la búsqueda en cuadrícula.

    """
    return rf_val.best_estimator_

<details>
<summary><h4><strong>Pista 1</strong></h4></summary>

Utilice `GridSearchCV()`, pasando en `rf` y `cv_params` y especificando `cv` como `custom_split`. Los argumentos adicionales que puede especificar incluyen: `refit='f1', n_jobs = -1, verbose = 1`. 

</details>


Ahora, ajusta tu modelo.


In [22]:
%%time

# Fit the model.

### YOUR CODE HERE ###


rf_val.fit(X_train, y_train)

Fitting 1 folds for each of 32 candidates, totalling 32 fits
CPU times: total: 8.44 s
Wall time: 1min 5s


,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",RandomForestC...andom_state=0)
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","{'max_depth': [10, 50], 'max_features': ['sqrt'], 'max_samples': [0.5, 0.9], 'min_samples_leaf': [0.5, 1], ...}"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion ` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.",None
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``GridSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`sphx_glr_auto_examples_model_selection_plot_grid_search_digits.py`to see how to design a custom selection strategy using a callablevia `refit`.See :ref:`this example`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",'f1'
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- An iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide ` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.","PredefinedSpl...ape=(97115,)))"
,"verbose verbose: intControls the verbosity: the higher, the more messages.- >1 : the computation time for each

<details>
<summary><h4><strong>Pista 1</strong></h4></summary>

Use el método `fit()` para entrenar el modelo GridSearchCV en `X_train` y `y_train`. 

</details>


<details>
<summary><h4><strong>Pista 2</strong></h4></summary>

Agrega la función mágica `%%time` para realizar un seguimiento de la cantidad de tiempo que tarda en ajustarse el modelo y mostrar esta información una vez que la ejecución haya finalizado. Recuerda que este código debe ser la primera línea en la celda.

</details>


Finalmente, obtener los parámetros óptimos.


In [23]:
# Obtain optimal parameters.

### YOUR CODE HERE ###

rf_val.best_params_

{'max_depth': 50,
 'max_features': 'sqrt',
 'max_samples': 0.9,
 'min_samples_leaf': 1,
 'min_samples_split': 0.001,
 'n_estimators': 50}

<details>
<summary><h4><strong>Pista 1</strong></h4></summary>

Use el atributo `best_params_` para obtener los valores óptimos de los hiperparámetros del modelo GridSearchCV.

</details>


## **Paso 4: Resultados y evaluación**


Use el modelo seleccionado para predecir en sus datos de prueba. Use los parámetros óptimos encontrados mediante GridSearchCV.


In [ ]:
# Use optimal parameters on GridSearchCV.

### YOUR CODE HERE ###

rf_opt = RandomForestClassifier(
    n_estimators = 50,           # numero de árboles en el bosque 
    max_depth = 50,              # profundidad máxima de cada árbol
    min_samples_leaf = 1,        # número mínimo de muestras en un nodo hoja
    min_samples_split = 0.001,   # número mínimo de muestras para dividir un nodo
    max_features="sqrt",         # número de características a considerar al buscar la mejor división
    max_samples = 0.9,           # fracción de muestras a utilizar para entrenar cada árbol
    random_state = 0)            # semilla para la generación de números aleatorios

<details>
<summary><h4><strong>Pista 1</strong></h4></summary>

Use `RandomForestClassifier()`, especificando el argumento `random_state` para resultados reproducibles y pasando los hiperparámetros óptimos encontrados en el paso anterior. Para distinguir esto del modelo de bosque aleatorio anterior, considere nombrar esta variable `rf_opt`.

</details>


Una vez más, ajusta el modelo óptimo.


In [ ]:
# Fit the optimal model.

### YOUR CODE HERE ###
# en este caso, el modelo óptimo es el mismo que el modelo de búsqueda en cuadrícula, 
# por lo que podemos usar el mejor estimador directamente
rf_opt.fit(X_train, y_train)

,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",50
,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.Note: This parameter is tree-specific.",'gini'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",50
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",0.001
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=""sqrt""The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to `""sqrt""`.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",'sqrt'
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow trees with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsamples at the current node, ``N_t_L`` is the number of samples in theleft child, and ``N_t_R`` is the number of samples in the right child.``N``, ``N_t``, ``N_t_R`` and ``N_t_L`` all refer to the weighted sum,if ``sample_weight`` is passed... versionadded:: 0.19",0.0
,"bootstrap bootstrap: bool, default=TrueWhether bootstrap samples are used when building trees. If False, thewhole dataset is used to build each tree.",True
,"oob_score oob_score: bool or callable, default=FalseWhether to use out-of-bag samples to estimate the generalization score.By default, :func:`~sklearn.metrics.accuracy_score` is used.Provide a callable with signature `metri

<details>
<summary><h4><strong>Pista 1</strong></h4></summary>

Use el método `fit()` para entrenar `rf_opt` en `X_train` y `y_train`.

</details>


Y predecir en el conjunto de prueba usando el modelo óptimo.


In [26]:
# Predict on test set.

### YOUR CODE HERE ###

y_pred = rf_opt.predict(X_test)

<details>
<summary><h4><strong>Pista 1</strong></h4></summary>

Puedes llamar a la función `predict()` para hacer predicciones en `X_test` usando `rf_opt`. Guarda las predicciones ahora (por ejemplo, como `y_pred`), para usarlas más tarde para comparar con las etiquetas verdaderas. 

</details>


### Obtener puntuaciones de rendimiento


Primero, obtén tu puntuación de precisión.


In [27]:
# Get precision score.

### YOUR CODE HERE ###

pc_test = precision_score(y_test, y_pred, pos_label = "satisfied")
# print("The precision score is {pc:.3f}".format(pc = pc_test))
print("la precisión del modelo es de {pc:.3f}".format(pc = pc_test))

la precisión del modelo es de 0.950


<details>
<summary><h4><strong>Pista 1</strong></h4></summary>

Puedes llamar a la función `precision_score()` de `sklearn.metrics`, pasando `y_test` y `y_pred` y especificando el argumento `pos_label` como `"satisfied"`.
</details>


Luego, recopila la puntuación de recuperación.


In [28]:
# Get recall score.

### YOUR CODE HERE ###

rc_test = recall_score(y_test, y_pred, pos_label = "satisfied")
# print("The recall score is {rc:.3f}".format(rc = rc_test))
print("la recuperación del modelo es de {rc:.3f}".format(rc = rc_test))

la recuperación del modelo es de 0.945


<details>
<summary><h4><strong>Pista 1</strong></h4></summary>

Puedes llamar a la función `recall_score()` de `sklearn.metrics`, pasando `y_test` y `y_pred` y especificando el argumento `pos_label` como `"satisfied"`.
</details>


Next, obtain your accuracy score.


In [29]:
# Get accuracy score.

### YOUR CODE HERE ###

ac_test = accuracy_score(y_test, y_pred)
print("The accuracy score is {ac:.3f}".format(ac = ac_test))
print("la exactitud del modelo es de {ac:.3f}".format(ac = ac_test))

The accuracy score is 0.942
la exactitud del modelo es de 0.942


<details>
<summary><h4><strong>Pista 1</strong></h4></summary>

Puedes llamar a la función `accuracy_score()` de `sklearn.metrics`, pasando `y_test` y `y_pred` y especificando el argumento `pos_label` como `"satisfied"`.
</details>


Finalmente, recopila tu puntuación F1.


In [30]:
# Get F1 score.

### YOUR CODE HERE ###

f1_test = f1_score(y_test, y_pred, pos_label = "satisfied")
print("The F1 score is {f1:.3f}".format(f1 = f1_test))
print("la puntuación F1 del modelo es de {f1:.3f}".format(f1 = f1_test))

The F1 score is 0.947
la puntuación F1 del modelo es de 0.947


<details>
<summary><h4><strong>Pista 1</strong></h4></summary>

Puedes llamar a la función `f1_score()` de `sklearn.metrics`, pasando `y_test` y `y_pred` y especificando el argumento `pos_label` como `"satisfied"`.
</details>


**Pregunta:** ¿Cómo se calcula la puntuación F1?

Las puntuaciones F1 se calculan usando la siguiente fórmula:

$$
F1 = 2 * \frac{precisión * recall}{precisión + recall}
$$

**¿Cuáles son las ventajas y desventajas de realizar la selección del modelo usando datos de prueba en lugar de un conjunto de validación separado?**

Pros: <br />
*  La carga de trabajo de codificación se reduce.
*  Los scripts para dividir los datos son más cortos.
*  Solo es necesario evaluar el rendimiento del conjunto de datos de prueba una vez, en lugar de dos evaluaciones (validar y probar).

Contras: <br />
* Si un modelo se evalúa usando muestras que también se usaron para construir o ajustar ese modelo, probablemente proporcionará una evaluación sesgada.
* Podría ocurrir un problema potencial de sobreajuste al ajustar las puntuaciones del modelo en los datos de prueba.


### Evaluar el modelo

Ahora que tienes resultados, evalúa el modelo.


Pregunta: ¿Cuáles son los cuatro parámetros básicos para evaluar el rendimiento de un modelo de clasificación?

1. Verdaderos positivos (TP): Estos son valores positivos predichos correctamente, lo que significa que el valor de las clases reales y predichas son positivos.

2. Verdaderos negativos (TN): Estos son valores negativos predichos correctamente, lo que significa que el valor de las clases reales y predichas son negativos.

3. Falsos positivos (FP): Esto ocurre cuando el valor de la clase real es negativo y el valor de la clase predicha es positivo.

4. Falsos negativos (FN): Esto ocurre cuando el valor de la clase real es positivo y el valor de la clase predicha es negativo.

**Recordatorio:** Cuando se ajustan y afinan modelos de clasificación, los profesionales de datos buscan minimizar los falsos positivos y falsos negativos.


**Pregunta:** ¿Qué demuestran los cuatro puntajes sobre tu modelo, y cómo los calculas?

- Precisión (VP+VN/VP+FP+FN+VN): La proporción de observaciones correctamente predichas en relación con el total de observaciones.

- Precisión (VP/VP+FP): La proporción de observaciones positivas correctamente predichas en relación con el total de observaciones positivas predichas.

- Recall (Sensibilidad, VP/VP+FN): La proporción de observaciones positivas correctamente predichas en relación con todas las observaciones en la clase real.

- Puntaje F1: El promedio armónico de precisión y recall, que tiene en cuenta tanto los falsos positivos como los falsos negativos.


Calcular las puntuaciones: puntuación de precisión, puntuación de recuperación, puntuación de exactitud, puntuación F1.


In [ ]:
# Precision score on test data set.

### YOUR CODE HERE ###

# print("\nThe precision score is: {pc:.3f}".format(pc = pc_test), "for the test set,", "\nwhich means of all positive predictions,", "{pc_pct:.1f}% prediction are true positive.".format(pc_pct = pc_test * 100))
print(f"La puntuación de precisión es: {pc_test:.3f} para el conjunto de prueba, lo que significa que de todas las predicciones positivas, {pc_test * 100:.1f}% son verdaderos positivos.")




The precision score is: 0.950 for the test set, 
which means of all positive predictions, 95.0% prediction are true positive.
La puntuación de precisión es: 0.950 para el conjunto de prueba, lo que significa que de todas las predicciones positivas, 95.0% son verdaderos positivos.


In [33]:
# Recall score on test data set.

### YOUR CODE HERE ###

# print("\nThe recall score is: {rc:.3f}".format(rc = rc_test), "for the test set,", "\nwhich means of which means of all real positive cases in test set,", "{rc_pct:.1f}% are  predicted positive.".format(rc_pct = rc_test * 100))
print(f"\nLa puntuación de recall es: {rc_test:.3f} para el conjunto de prueba, lo que significa que de todos los casos positivos reales en el conjunto de prueba, {rc_test * 100:.1f}% se predicen como positivos.")


La puntuación de recall es: 0.945 para el conjunto de prueba, lo que significa que de todos los casos positivos reales en el conjunto de prueba, 94.5% se predicen como positivos.


In [34]:
# Accuracy score on test data set.

### YOUR CODE HERE ###

# print("\nThe accuracy score is: {ac:.3f}".format(ac = ac_test), "for the test set,", "\nwhich means of all cases in test set,", "{ac_pct:.1f}% are predicted true positive or true negative.".format(ac_pct = ac_test * 100))
print(f"\nLa puntuación de precisión es: {ac_test:.3f} para el conjunto de prueba, lo que significa que de todos los casos en el conjunto de prueba, {ac_test * 100:.1f}% se predicen como verdaderos positivos o verdaderos negativos.")


La puntuación de precisión es: 0.942 para el conjunto de prueba, lo que significa que de todos los casos en el conjunto de prueba, 94.2% se predicen como verdaderos positivos o verdaderos negativos.


In [35]:
# F1 score on test data set.

### YOUR CODE HERE ###

# print("\nThe F1 score is: {f1:.3f}".format(f1 = f1_test), "for the test set,", "\nwhich means the test set's harmonic mean is {f1_pct:.1f}%.".format(f1_pct = f1_test * 100))
print(f"\nLa puntuación F1 es: {f1_test:.3f} para el conjunto de prueba, lo que significa que la media armónica del conjunto de prueba es {f1_test * 100:.1f}%.")


La puntuación F1 es: 0.947 para el conjunto de prueba, lo que significa que la media armónica del conjunto de prueba es 94.7%.


**Pregunta:** ¿Cómo funciona este modelo según las cuatro puntuaciones?

El modelo funciona bien según las 4 métricas de rendimiento. La puntuación de precisión del modelo es ligeramente mejor que las otras 3 métricas.


### Evaluar el modelo

Finalmente, crea una tabla de resultados que puedas usar para evaluar el rendimiento de tu modelo.


In [31]:
# Create table of results.

### YOUR CODE HERE ###
table = pd.DataFrame({'Model': ["Tuned Decision Tree", "Tuned Random Forest"],
                        'F1':  [0.945422, f1_test],
                        'Recall': [0.935863, rc_test],
                        'Precision': [0.955197, pc_test],
                        'Accuracy': [0.940864, ac_test]
                      }
                    )
table

,Model,F1,Recall,Precision,Accuracy
0,Tuned Decision Tree,0.945422,0.935863,0.955197,0.940864
1,Tuned Random Forest,0.947306,0.944501,0.950128,0.942450


<details>
<summary><h4><strong>Pista 1</strong></h4></summary>

Construye una tabla para comparar el rendimiento de los modelos. Crea un DataFrame usando la función `pd.DataFrame()`.

</details>


**Pregunta:** ¿Cómo se compara el modelo de bosque aleatorio con el modelo de árbol de decisión que construiste en el laboratorio anterior?

El bosque aleatorio ajustado tiene puntuaciones más altas en general, por lo que es el mejor modelo. Particularmente, muestra una mejor puntuación F1 que el modelo de árbol de decisión, lo que indica que el modelo de bosque aleatorio puede hacerlo mejor en clasificación al tener en cuenta falsos positivos y falsos negativos.


## **Consideraciones**


**¿Cuáles son las conclusiones clave de este laboratorio?**
- La exploración, limpieza y codificación de datos son necesarias para la construcción del modelo.
- Un conjunto de validación separado se usa típicamente para ajustar un modelo, en lugar de usar el conjunto de prueba. Esto también ayuda a evitar que la evaluación se vuelva sesgada.
- Las puntuaciones F1 suelen ser más útiles que las puntuaciones de precisión. Si el costo de falsos positivos y falsos negativos es muy diferente, es mejor usar la puntuación F1 y combinar la información de precisión y recall.
* El modelo de bosque aleatorio ofrece un rendimiento más efectivo que un modelo de árbol de decisión.

**¿Qué resumen proporcionarías a las partes interesadas?**
* El modelo de bosque aleatorio predijo la satisfacción con una precisión de más del 94.2%. La precisión es superior al 95% y el recall es aproximadamente 94.5%.
* El modelo de bosque aleatorio superó al árbol de decisión ajustado con los mejores hiperparámetros en la mayoría de las cuatro puntuaciones. Esto indica que el modelo de bosque aleatorio puede tener un mejor rendimiento.
* Debido a que las partes interesadas estaban interesadas en aprender sobre los factores que son más importantes para la satisfacción del cliente, esto se compartiría basado en el bosque aleatorio ajustado.
* Además, proporcionarías detalles sobre las puntuaciones de precisión, recall, exactitud y F1 para respaldar tus hallazgos.


### Referencias


¿Qué es la diferencia entre conjuntos de datos de prueba y validación?, Jason Brownlee

Decision Trees and Random Forests Neil Liberman


**¡Felicidades!** Has completado este laboratorio. Sin embargo, es posible que no notes una marca de verificación verde junto a este elemento en la plataforma de Coursera. Por favor, continúa con tu progreso independientemente de la marca de verificación. Solo haz clic en el icono de "guardar" en la parte superior de este cuaderno para asegurarte de que tu trabajo ha sido registrado
